#Assignment-4


Ques-1
You are given a large NumPy array of size 5000000 ini alized with random values. Compute the following element-wise opera on: f(x)=x2+3x+5, for each element in the array and convert it into a CUDA kernel using Numba. Compare performance difference of CPU with GPU. a. Modify the kernel to use float32 and float64


##with numpy (CPU)


In [4]:
import numpy as np
import time
N=5000000
arr=np.random.rand(N)
start=time.time()
f_x=arr**2+3*arr+5
cpu_time=time.time() - start
print("CPU Time:", cpu_time)

CPU Time: 0.03719782829284668


## With Numba float 32

In [5]:
from numba import cuda
import time

N=5000000
arr=np.random.rand(N).astype(np.float32)
d_arr=cuda.to_device(arr)
d_out=cuda.device_array_like(arr)

@cuda.jit
def kernel32(a, out):
    i=cuda.grid(1)
    if i<a.size:
        x =a[i]
        out[i]=x*x + 3*x + 5

threads=256
blocks=(N+threads-1)//threads

start=time.time()

kernel32[blocks, threads](d_arr, d_out)
cuda.synchronize()
gpu32_time=time.time() - start
print("GPU Time (float32):", gpu32_time)

GPU Time (float32): 0.06755948066711426


##With Numba float 64

In [6]:
from numba import cuda
import time

N=5000000
arr=np.random.rand(N).astype(np.float64)
d_arr=cuda.to_device(arr)
d_out=cuda.device_array_like(arr)

@cuda.jit
def kernel64(a, out):
    i=cuda.grid(1)
    if i<a.size:
        x =a[i]
        out[i]=x*x + 3*x + 5

threads=256
blocks=(N+threads-1)//threads

start=time.time()

kernel64[blocks, threads](d_arr, d_out)
cuda.synchronize()
gpu64_time=time.time() - start
print("GPU Time (float64):", gpu64_time)

GPU Time (float64): 0.0889289379119873


in this cpu is showing better performance than gpu for simple numpy operations

Ques-2 Implement and benchmark a 1-D histogram computa on for 1 million random values in Python using Numba. Compare different approaches (pure Python, NumPy, and Numba-accelerated) and analyze performance and correctness.

In [13]:
import numpy as np
import time
from numba import njit

N=1_000_000
bins=20
data=np.random.rand(N)

In [14]:
start=time.time()
hist_py=[0]*bins

for x in data:
    idx=int(x*bins)
    if idx==bins:
        idx=bins-1
    hist_py[idx]+=1

py_time=time.time()-start
print("Pure Python time:",py_time)

Pure Python time: 0.3826169967651367


In [15]:
start=time.time()
hist_np,edges=np.histogram(data,bins=bins,range=(0,1))
numpy_time=time.time()-start

print("NumPy time:",numpy_time)

NumPy time: 0.018570423126220703


In [16]:
@njit
def histogram_numba(data,bins):

    hist=np.zeros(bins,dtype=np.int64)

    for i in range(len(data)):
        x=data[i]
        idx=int(x*bins)
        if idx==bins:
            idx=bins-1

        hist[idx]+=1

    return hist


start=time.time()
hist_nb=histogram_numba(data,bins)
numba_time=time.time()-start

print("Numba time:",numba_time)

Numba time: 0.11532998085021973


In [17]:
print("Python vs NumPy:",np.all(hist_py==hist_np))
print("NumPy vs Numba:",np.all(hist_np==hist_nb))

Python vs NumPy: True
NumPy vs Numba: True


Ques-3  Write a func on monte_carlo_pi(nsamples) that esmates the value of π by genera ng random x, y coordinates between 0 and 1 and checking if they fall inside a unit circle (x2 + y2 < 1).

a. Implement the func on in pure Python first and later create a Numba version.  

b. Program a script to compare the execu on me for 5 million samples. Report the Speedup Factor (Python Time / Numba Time).

c. Why does the very first execu on of the Numba func on take slightly longer than the second execu on?

In [18]:
import random
import time

def monte_carlo_pi(nsamples):
    inside=0
    for i in range(nsamples):
        x=random.random()
        y=random.random()
        if x*x+y*y<1:
            inside+=1
    return 4*inside/nsamples


N=5000000
start=time.time()
pi_py=monte_carlo_pi(N)
python_time=time.time()-start
print("Python π estimate:",pi_py)
print("Python time:",python_time)

Python π estimate: 3.1424912
Python time: 0.9256627559661865


In [19]:
from numba import njit
import numpy as np
import time

@njit
def monte_carlo_pi_numba(nsamples):

    inside=0

    for i in range(nsamples):
        x=np.random.random()
        y=np.random.random()

        if x*x+y*y<1:
            inside+=1

    return 4*inside/nsamples

In [20]:
N=5_000_000

start=time.time()
pi_nb=monte_carlo_pi_numba(N)
numba_time=time.time()-start

print("Numba π estimate:",pi_nb)
print("Numba time:",numba_time)

Numba π estimate: 3.1430832
Numba time: 0.18282508850097656


In [21]:
speedup=python_time/numba_time
print("Speedup Factor:",speedup)

Speedup Factor: 5.063105745255756


##The first execution of a Numba function is slower because Numba performs Just-In-Time compilation during the first call. After compilation, subsequent executions reuse the compiled machine code, resulting in faster execution.

Ques-4  You have a 1D NumPy array representing pixel intensities (values 0–255). You need to increase the brightness of every pixel by 20%, but ensure no value exceeds 255.

a. Write a function adjust_brightness(pixel_value) using the @vectorize decorator.

b. Apply this function to an array of 10 million random integers.

c. Change the decorator to @vectorize(['int64(int64)'], target='parallel'). Measure the me difference when the work is automatically split across your CPU cores.

d. What happens if you try to pass a list instead of a NumPy array to this function?

In [22]:
import numpy as np
from numba import vectorize

@vectorize
def adjust_brightness(pixel_value):
    new_value=int(pixel_value*1.2)
    if new_value>255:
        return 255
    else:
        return new_value

In [28]:
N=10000000
pixels=np.random.randint(0,256,N)
bright_pixels=adjust_brightness(pixels)
print(bright_pixels[:10])

start=time.time()
bright_pixels=adjust_brightness(pixels)
normal_time=time.time()-start
print("Normal vectorize time:",normal_time)

[204 247 118 252 243 108 255 255 229 163]
Normal vectorize time: 0.03494429588317871


Parallel version using cpu cores


In [29]:
from numba import vectorize
import time

@vectorize(['int64(int64)'],target='parallel')
def adjust_brightness_parallel(pixel_value):
    new_value=int(pixel_value*1.2)
    if new_value>255:
        return 255
    else:
        return new_value

start=time.time()
bright_pixels_parallel=adjust_brightness_parallel(pixels)
parallel_time=time.time()-start
print("Parallel execution time:",parallel_time)

Parallel execution time: 0.030417919158935547


In [30]:
pixel_list=[10,50,100,200]
result=adjust_brightness(pixel_list)
print(result)

[ 12  60 120 240]


Ques-5 Write Python code to generate synthe c training data of 100,000 samples, 10 features and binary labels {-1, +1}. Implement binary logis c regression using the mathema cal formula for gradient descent:

a. Using standard NumPy (without Numba)

b. Using Numba JIT acceleration

c. Compare correctness and performance.

In [37]:
import numpy as np
np.random.seed(0)
N=100000
features=10
X=np.random.randn(N,features)
true_w=np.random.randn(features)
y=np.sign(X @ true_w)
y[y==0]=1

import time

def logistic_regression_numpy(X,y,lr=0.01,iterations=100):

    N,features=X.shape
    w=np.zeros(features)
    for i in range(iterations):
        z=X @ w
        gradient=-(X.T @ (y/(1+np.exp(y*z))))/N
        w=w-lr*gradient
    return w


start=time.time()
w_numpy=logistic_regression_numpy(X,y)
numpy_time=time.time()-start
print("NumPy time:",numpy_time)

NumPy time: 0.37551140785217285


In [49]:
from numba import njit

@njit
def logistic_regression_numba(X,y,lr=0.01,iterations=100):

    N=X.shape[0]
    features=X.shape[1]
    w=np.zeros(features)

    for it in range(iterations):
        gradient=np.zeros(features)
        for i in range(N):
            dot=0.0
            for j in range(features):
                dot+=X[i,j]*w[j]
            coeff=y[i]/(1+np.exp(y[i]*dot))

            for j in range(features):
                gradient[j]-=X[i,j]*coeff

        for j in range(features):
            gradient[j]/=N
            w[j]-=lr*gradient[j]

    return w


In [55]:
start=time.time()
w_numba=logistic_regression_numba(X,y)
numba_time=time.time()-start
print("Numba time:",numba_time)

Numba time: 0.2885129451751709


In [56]:
print("Weight difference:",np.linalg.norm(w_numpy-w_numba))

Weight difference: 1.8990320241572316e-16


In [57]:
print("NumPy time:",numpy_time)
print("Numba time:",numba_time)
print("Speedup:",numpy_time/numba_time)

NumPy time: 0.37551140785217285
Numba time: 0.2885129451751709
Speedup: 1.3015409330218468


Ques-6  Write a CUDA kernel to add two large matrices (A + B = C) of size 1024 X 1024.

In [60]:
import numpy as np
from numba import cuda

@cuda.jit
def matrix_add(A,B,C):
    row,col=cuda.grid(2)

    if row<A.shape[0] and col<A.shape[1]:
        C[row,col]=A[row,col]+B[row,col]


N=1024
A=np.random.rand(N,N)
B=np.random.rand(N,N)

d_A=cuda.to_device(A)
d_B=cuda.to_device(B)
d_C=cuda.device_array((N,N))

threads_per_block=(16,16)
blocks_x=(N+threads_per_block[0]-1)//threads_per_block[0]
blocks_y=(N+threads_per_block[1]-1)//threads_per_block[1]
blocks_per_grid=(blocks_x,blocks_y)

matrix_add[blocks_per_grid,threads_per_block](d_A,d_B,d_C)

C=d_C.copy_to_host()

print("Matrix addition completed")
print(C[:5,:5])

Matrix addition completed
[[0.63783178 1.25784774 0.91146569 0.67096221 0.71963715]
 [1.31253548 1.54264539 1.21266025 0.76229212 1.27306922]
 [0.72374428 0.69345742 1.80150102 1.27049608 0.24539351]
 [1.37426461 1.15398075 0.49598063 0.86473932 1.02135007]
 [0.93253189 1.01753305 0.85208057 1.48427525 1.35788166]]
